In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU in use:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


CUDA available: True
GPU in use: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [ ]:
import pandas as pd
import numpy as np
print("Hello World")
dataframe_A = pd.read_csv(r"combinedA_patient_id_cleaned.csv")
dataframe_B = pd.read_csv(r"combinedB_patient_id_cleaned.csv")
dataframe = dataframe_A
dataframe_testing = dataframe_B
dataframe_A.head()

Hello World


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,HCO3,FiO2,...,WBC,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,80.0,100.0,36.50,121.00,58.0,41.00,13.5,1.000000,25.000000,1.000000,...,9.900000,160.000000,77.27,1,0.0,1.0,-69.14,3,0,14977
1,76.0,100.0,36.25,113.25,61.0,41.50,12.0,1.000000,25.000000,0.500000,...,9.900000,199.617841,77.27,1,0.0,1.0,-69.14,4,0,14977
2,80.0,100.0,36.25,132.75,71.5,46.25,12.0,-0.647537,24.094476,0.526248,...,11.936604,199.617841,77.27,1,0.0,1.0,-69.14,5,0,14977
3,78.0,100.0,36.10,103.50,58.0,43.00,12.0,-3.000000,24.094476,0.526248,...,11.936604,199.617841,77.27,1,0.0,1.0,-69.14,6,0,14977
4,74.0,100.0,36.00,128.75,69.5,44.50,12.5,-3.000000,24.094476,0.526248,...,11.936604,199.617841,77.27,1,0.0,1.0,-69.14,7,0,14977


In [16]:
dataframe_A.shape
dataframe_A.isna().sum()
dataframe_A.sort_values(by=["patient_id", "ICULOS"])


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,HCO3,FiO2,...,WBC,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
401726,84.985264,97.265688,37.026737,120.962359,78.767345,59.985809,18.77346,-0.647537,24.094476,0.526248,...,11.936604,199.617841,83.14,0,0.507101,0.492899,-0.03,1,0,1
401727,97.000000,95.000000,37.026737,98.000000,75.330000,59.985809,19.00000,-0.647537,24.094476,0.526248,...,11.936604,199.617841,83.14,0,0.507101,0.492899,-0.03,2,0,1
401728,89.000000,99.000000,37.026737,122.000000,86.000000,59.985809,22.00000,-0.647537,24.094476,0.526248,...,11.936604,199.617841,83.14,0,0.507101,0.492899,-0.03,3,0,1
401729,90.000000,95.000000,37.026737,120.962359,78.767345,59.985809,30.00000,24.000000,24.094476,0.526248,...,11.936604,199.617841,83.14,0,0.507101,0.492899,-0.03,4,0,1
401730,103.000000,88.500000,37.026737,122.000000,91.330000,59.985809,24.50000,-0.647537,24.094476,0.280000,...,11.936604,199.617841,83.14,0,0.507101,0.492899,-0.03,5,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207131,88.000000,98.000000,37.026737,135.000000,81.000000,64.000000,16.00000,-0.647537,24.094476,0.500000,...,11.936604,199.617841,62.29,1,0.507101,0.492899,-0.03,31,1,20643
207132,96.000000,98.000000,38.720000,174.000000,97.000000,72.000000,16.00000,2.000000,24.094476,0.526248,...,11.936604,199.617841,62.29,1,0.507101,0.492899,-0.03,32,1,20643
207133,140.000000,97.000000,37.026737,133.000000,81.500000,62.500000,16.00000,-0.647537,24.094476,0.526248,...,11.936604,199.617841,62.29,1,0.507101,0.492899,-0.03,33,1,20643
207134,120.000000,96.000000,37.026737,154.000000,118.000000,105.000000,16.00000,-0.647537,24.094476,0.526248,...,11.936604,199.617841,62.29,1,0.507101,0.492899,-0.03,34,1,20643


In [17]:
def split_data(dataframe, patient_column, time_column):
    patient_groups = dataframe.groupby(patient_column)
    sequences = []

    for patient_id, group in patient_groups:
        group = group.sort_values(by=time_column)
        sequences.append(group)

    return sequences



In [ ]:

SPARSE_FEATURES = {
    "Temp", "DBP",
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "BUN", "Calcium",
    "Chloride", "Creatinine", "Glucose", "Lactate", "Phosphate",
    "Potassium", "Hct", "Hgb", "WBC", "Platelets",
}


def find_imputation_fillers(dataframe, features, min_frac=0.35):
    """Detect constant fill values used for missing labs/vitals."""
    fillers = {}
    for col in features:
        if col not in SPARSE_FEATURES or col not in dataframe.columns:
            continue
        vc = dataframe[col].round(6).value_counts(normalize=True)
        if len(vc) and float(vc.iloc[0]) >= min_frac:
            fillers[col] = float(vc.index[0])
    return fillers


def calculate_mask(X, features, fillers):
    X = np.asarray(X, dtype=np.float32)
    M = np.ones_like(X, dtype=np.float32)
    for j, name in enumerate(features):
        if name in fillers:
            M[:, j] = (np.abs(X[:, j] - fillers[name]) > 1e-5).astype(np.float32)
    return M


In [ ]:
def calculate_decay(mask):
    m = np.asarray(mask, dtype=np.float32)
    T, F = m.shape
    t = np.arange(T, dtype=np.float32)[:, None]
    observed = m > 0
    last_t = np.where(observed, t, np.nan)
    # forward-fill last observation time
    last_t = pd.DataFrame(last_t).ffill().fillna(0.0).to_numpy(dtype=np.float32)
    decay = np.where(observed, 0.0, t - last_t).astype(np.float32)
    return decay


In [ ]:
def build_sequence(dataframe, features, patient_column, time_column):
    df = dataframe.sort_values([patient_column, time_column], kind="mergesort")
    fillers = find_imputation_fillers(df, features)
    print(f"  recovered missingness for {len(fillers)} features: {sorted(fillers)}")

    X_all = df[features].to_numpy(dtype=np.float32)
    y_all = df["SepsisLabel"].to_numpy(dtype=np.float32)
    patient_ids = df[patient_column].to_numpy()

    boundaries = np.flatnonzero(patient_ids[1:] != patient_ids[:-1]) + 1
    starts = np.r_[0, boundaries]
    ends = np.r_[boundaries, len(df)]

    X_list, y_list, M_list, D_list = [], [], [], []
    for s, e in zip(starts, ends):
        X = X_all[s:e]
        y = y_all[s:e]
        M = calculate_mask(X, features, fillers)
        D = calculate_decay(M)
        X_list.append(X)
        y_list.append(y)
        M_list.append(M)
        D_list.append(D)

    obs = np.mean([M.mean() for M in M_list[:200]])
    print(f"  approx observed fraction (sample): {obs:.3f}")
    return X_list, y_list, M_list, D_list


In [ ]:
def split_by_patient(X, M, D, y, test_size, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(X))
    rng.shuffle(idx)
    n_test = int(len(X) * test_size)
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]

    def take(lst, ids):
        return [lst[i] for i in ids]

    return (
        take(X, train_idx), take(M, train_idx), take(D, train_idx), take(y, train_idx),
        take(X, test_idx), take(M, test_idx), take(D, test_idx), take(y, test_idx),
    )


In [22]:
import time

features = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "BUN", "Calcium",
    "Chloride", "Creatinine", "Glucose", "Lactate", "Phosphate",
    "Potassium", "Hct", "Hgb", "WBC", "Platelets",
    "Age", "Gender", "HospAdmTime",
]

t0 = time.time()
print("Building Hospital A sequences...")
X_A, y_A, M_A, D_A = build_sequence(dataframe_A, features, "patient_id", "ICULOS")
print("Building Hospital B sequences...")
X_B, y_B, M_B, D_B = build_sequence(dataframe_B, features, "patient_id", "ICULOS")
print(f"done in {time.time() - t0:.1f}s | A: {len(X_A)} patients | B: {len(X_B)} patients")


Building Hospital A sequences...


  recovered missingness for 19 features: ['BUN', 'BaseExcess', 'Calcium', 'Chloride', 'Creatinine', 'DBP', 'FiO2', 'Glucose', 'HCO3', 'Hct', 'Hgb', 'Lactate', 'PaCO2', 'Phosphate', 'Platelets', 'Potassium', 'Temp', 'WBC', 'pH']
  approx observed fraction (sample): 0.384
Building Hospital B sequences...
  recovered missingness for 18 features: ['BUN', 'BaseExcess', 'Calcium', 'Chloride', 'Creatinine', 'FiO2', 'Glucose', 'HCO3', 'Hct', 'Hgb', 'Lactate', 'PaCO2', 'Phosphate', 'Platelets', 'Potassium', 'Temp', 'WBC', 'pH']
  approx observed fraction (sample): 0.375
done in 26.0s | A: 20336 patients | B: 20000 patients


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


class GRUDModel(nn.Module):
    """GRU-D style: decay + mask-aware input, per-timestep logits."""

    def __init__(self, input_size, hidden_size=128, output_size=1, dropout=0.2):
        super().__init__()
        num_layers = 2
        self.gamma_x = nn.Parameter(torch.zeros(input_size))
        self.gamma_h = nn.Parameter(torch.zeros(1))
        self.input_proj = nn.Linear(input_size * 2, hidden_size)
        
        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, X, M, D, mean_values):
        mean_values = mean_values.view(1, 1, -1)
        gamma_x = torch.exp(-F.softplus(self.gamma_x) * D)
        X_hat = M * X + (1.0 - M) * (gamma_x * X + (1.0 - gamma_x) * mean_values)
        h = torch.tanh(self.input_proj(torch.cat([X_hat, M], dim=-1)))
        out, _ = self.gru(h)
        gamma_h = torch.exp(-F.softplus(self.gamma_h) * D.mean(dim=-1, keepdim=True))
        out = self.dropout(gamma_h * out)
        return self.output_layer(out)


Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [24]:
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import roc_auc_score

BATCH_SIZE = 32
EPOCHS = 35
HIDDEN = 256
DROPOUT = 0.2
LR = 3e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def collate_patient_batch(X_list, M_list, D_list, y_list, indices):
    xs, ms, ds, ys, lengths = [], [], [], [], []
    for i in indices:
        xs.append(torch.tensor(X_list[i], dtype=torch.float32))
        ms.append(torch.tensor(M_list[i], dtype=torch.float32))
        ds.append(torch.tensor(D_list[i], dtype=torch.float32))
        ys.append(torch.tensor(y_list[i], dtype=torch.float32))
        lengths.append(len(y_list[i]))
    return (
        pad_sequence(xs, batch_first=True).to(device),
        pad_sequence(ms, batch_first=True).to(device),
        pad_sequence(ds, batch_first=True).to(device),
        pad_sequence(ys, batch_first=True).to(device),
        torch.tensor(lengths, dtype=torch.long),
    )


def standardize(X_train, X_test):
    cat = np.concatenate(X_train, axis=0)
    mean = cat.mean(axis=0).astype(np.float32)
    std = (cat.std(axis=0) + 1e-6).astype(np.float32)
    return [(x - mean) / std for x in X_train], [(x - mean) / std for x in X_test]


def train_grud(X_tr, M_tr, D_tr, y_tr, epochs=EPOCHS):
    y_flat = np.concatenate(y_tr)
    n_pos = max(int((y_flat == 1).sum()), 1)
    n_neg = int((y_flat == 0).sum())
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)

    model = GRUDModel(len(features), hidden_size=HIDDEN, dropout=DROPOUT).to(device)
    criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    mean_values = torch.zeros(len(features), dtype=torch.float32, device=device)

    n = len(X_tr)
    model.train()
    for _ in range(epochs):
        order = np.random.permutation(n)
        for start in range(0, n, BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            X_b, M_b, D_b, y_b, lengths = collate_patient_batch(X_tr, M_tr, D_tr, y_tr, idx)
            opt.zero_grad()
            preds = model(X_b, M_b, D_b, mean_values).squeeze(-1)
            loss = criterion(preds, y_b)
            mask = torch.arange(preds.size(1), device=device).unsqueeze(0) < lengths.to(device).unsqueeze(1)
            loss = (loss * mask).sum() / mask.sum().clamp_min(1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
    return model, mean_values


def eval_roc_auc(model, mean_values, X_te, M_te, D_te, y_te):
    """Patient-level ROC-AUC: score = max prob in stay, label = any sepsis."""
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for start in range(0, len(X_te), BATCH_SIZE):
            idx = list(range(start, min(start + BATCH_SIZE, len(X_te))))
            X_b, M_b, D_b, y_b, lengths = collate_patient_batch(X_te, M_te, D_te, y_te, idx)
            probs = torch.sigmoid(model(X_b, M_b, D_b, mean_values).squeeze(-1))
            for i, L in enumerate(lengths.tolist()):
                scores.append(float(probs[i, :L].max().cpu()))
                labels.append(int(y_b[i, :L].max().cpu() > 0))
    return float(roc_auc_score(labels, scores))


def run_experiment(X_src, M_src, D_src, y_src, X_tgt=None, M_tgt=None, D_tgt=None, y_tgt=None, internal=False):
    if internal:
        X_tr, M_tr, D_tr, y_tr, X_te, M_te, D_te, y_te = split_by_patient(
            X_src, M_src, D_src, y_src, 0.2, seed=SEED
        )
    else:
        X_tr, M_tr, D_tr, y_tr = X_src, M_src, D_src, y_src
        X_te, M_te, D_te, y_te = X_tgt, M_tgt, D_tgt, y_tgt

    X_tr_s, X_te_s = standardize(X_tr, X_te)
    model, mean_values = train_grud(X_tr_s, M_tr, D_tr, y_tr)
    return eval_roc_auc(model, mean_values, X_te_s, M_te, D_te, y_te)

print("helpers ready")


helpers ready


In [ ]:
roc_AA = run_experiment(X_A, M_A, D_A, y_A, internal=True)
roc_BB = run_experiment(X_B, M_B, D_B, y_B, internal=True)
roc_AB = run_experiment(X_A, M_A, D_A, y_A, X_B, M_B, D_B, y_B, internal=False)
roc_BA = run_experiment(X_B, M_B, D_B, y_B, X_A, M_A, D_A, y_A, internal=False)

print(f"+ Train A, test A: {roc_AA:.3f} ROC AUC")
print(f"+ Train B, test B: {roc_BB:.3f} ROC AUC")
print(f"+ Train A, test B (cross hospital): {roc_AB:.3f} ROC AUC")
print(f"+ Train B, test A (cross hospital): {roc_BA:.3f} ROC AUC")


+ Train A, test A: 0.735 ROC AUC
+ Train B, test B: 0.643 ROC AUC
+ Train A, test B (cross hospital): 0.635 ROC AUC
+ Train B, test A (cross hospital): 0.654 ROC AUC


In [26]:
# Main results are printed in the previous cell.
